# HW - 6

I had used Claude for assistance

# Problem 1.

### Breadth-First Search (BFS) using a queue

    def countIslandsUsingBFS(self, grid):
        if not grid or not grid[0]:
            return 0

        rows, cols = len(grid), len(grid[0])
        g = [row[:] for row in grid]
        count = 0
        directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]

        for r in range(rows):
            for c in range(cols):
                if g[r][c] == '1':
                    count += 1
                    queue = deque([(r, c)])
                    g[r][c] = '0'  # mark visited when enqueueing
                    while queue:
                        cr, cc = queue.popleft()
                        for dr, dc in directions:
                            nr, nc = cr + dr, cc + dc
                            if 0 <= nr < rows and 0 <= nc < cols and g[nr][nc] == '1':
                                g[nr][nc] = '0'
                                queue.append((nr, nc))
        return count

### Depth-First Search (DFS) using recursion

    def countIslandsUsingDFS(self, grid):
        if not grid or not grid[0]:
            return 0

        rows, cols = len(grid), len(grid[0])
        # Make a deep-ish copy so we don't mutate the caller's grid
        # (the test harness reuses the same grid across all 3 methods).
        g = [row[:] for row in grid]
        count = 0

        def dfs(r, c):
            # Out of bounds or water/visited -> stop
            if r < 0 or r >= rows or c < 0 or c >= cols or g[r][c] != '1':
                return
            g[r][c] = '0'  # mark visited by sinking the land
            dfs(r + 1, c)
            dfs(r - 1, c)
            dfs(r, c + 1)
            dfs(r, c - 1)

        for r in range(rows):
            for c in range(cols):
                if g[r][c] == '1':
                    count += 1
                    dfs(r, c)
        return count

### Depth-First Search (DFS) using a stack (iterative)

    def countIslandsUsingDFSUsingStack(self, grid):
        if not grid or not grid[0]:
            return 0

        rows, cols = len(grid), len(grid[0])
        g = [row[:] for row in grid]
        count = 0
        directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]

        for r in range(rows):
            for c in range(cols):
                if g[r][c] == '1':
                    count += 1
                    stack = [(r, c)]
                    g[r][c] = '0'  # mark visited when pushing
                    while stack:
                        cr, cc = stack.pop()
                        for dr, dc in directions:
                            nr, nc = cr + dr, cc + dc
                            if 0 <= nr < rows and 0 <= nc < cols and g[nr][nc] == '1':
                                g[nr][nc] = '0'
                                stack.append((nr, nc))
        return count

## Test with edge cases (e.g., empty grids, single-element grids, all-land or all-water)

In [1]:
from Homework6 import Homework6

hw = Homework6()

# Empty grid
print("Empty grid:")
print(hw.countIslandsUsingDFS([]))
print(hw.countIslandsUsingBFS([]))
print(hw.countIslandsUsingDFSUsingStack([]))

# Single land cell
print("\nSingle land cell:")
print(hw.countIslandsUsingDFS([['1']]))
print(hw.countIslandsUsingBFS([['1']]))
print(hw.countIslandsUsingDFSUsingStack([['1']]))

# Single water cell
print("\nSingle water cell:")
print(hw.countIslandsUsingDFS([['0']]))
print(hw.countIslandsUsingBFS([['0']]))
print(hw.countIslandsUsingDFSUsingStack([['0']]))

# All water
print("\nAll water:")
grid = [['0','0','0'],['0','0','0']]
print(hw.countIslandsUsingDFS(grid))
print(hw.countIslandsUsingBFS(grid))
print(hw.countIslandsUsingDFSUsingStack(grid))

# All land
print("\nAll land:")
grid = [['1','1','1'],['1','1','1']]
print(hw.countIslandsUsingDFS(grid))
print(hw.countIslandsUsingBFS(grid))
print(hw.countIslandsUsingDFSUsingStack(grid))

Empty grid:
0
0
0

Single land cell:
1
1
1

Single water cell:
0
0
0

All water:
0
0
0

All land:
1
1
1


## Mention any limitations of recursive DFS in your analysis

- The recursion depth equals the size of the largest island in the worst case. 

- Python's default recursion limit is 1000, so a single connected island larger than ~1000 cells (easy on, say, a 50×50 all-land grid) will hit RecursionError. The two iterative variants don't have this problem — they're bounded by heap memory, not stack frames.
-For competition or production code on potentially large grids, prefer BFS or stack-DFS; reserve recursive DFS for small inputs or when you're willing to bump sys.setrecursionlimit.

- Complexity for all three: O(rows × cols) time and space — every cell is visited once, and the frontier (call stack / queue / explicit stack) can hold up to O(rows × cols) cells in the degenerate all-land case.

# FOR THE LAST TIME (this semester) : thank you: D 

wait..there's also part-2 (almost missed it)

# Part II: Graph Traversal (DFS)

## Reading the graph

From the figure, the directed edges are:

a → c, a → d, c → b, b → f, d → e, e → g, f → g, f → i, g → i, h → f, i → h

Adjacency list:

- a: c, d
- b: f
- c: b
- d: e
- e: g
- f: g, i
- g: i
- h: f
- i: h


## Problem 1: DFS traversal

I'll start from **h** and explore neighbors in **reverse alphabetical order**. When DFS finishes from h, I restart from the next unvisited vertex (reverse-alpha).

Walking through it (using a clock that ticks on each discover/finish):

Start at h → d(h) = 1.
From h, go to f → d(f) = 2.
f's neighbors are g and i. Reverse-alpha picks i first → d(i) = 3.
From i, go to h. h is already visited → this is a **back edge**.
i has no other neighbors → f(i) = 4.
Back at f, next neighbor is g → d(g) = 5.
From g, go to i. i is already finished → **cross edge**.
g done → f(g) = 6.
f done → f(f) = 7.
h done → f(h) = 8.

Now restart. Unvisited: a, b, c, d, e. Reverse-alpha → start from e.

d(e) = 9. e → g, already finished → **cross edge**. f(e) = 10.

Next unvisited (reverse-alpha): d.
d(d) = 11. d → e, already finished → **cross edge**. f(d) = 12.

Next: c.
d(c) = 13. c → b → d(b) = 14. b → f, finished → **cross edge**. f(b) = 15. f(c) = 16.

Last: a.
d(a) = 17. a's neighbors c, d (reverse-alpha → d first). Both finished → both are **cross edges**. f(a) = 18.

## Problem 2: DFS forest and edge labels

The DFS forest has 5 trees (one per restart):

```
   h          e     d     c        a
   |                      |
   f                      b
  / \
 i   g
```

Tree edges (the ones DFS actually used to discover new vertices):
**h→f, f→i, f→g, c→b**

Now classifying every edge in the original graph:

| Edge  | Class | Why |
|-------|-------|-----|
| h → f | tree  | discovered f from h |
| f → i | tree  | discovered i from f |
| f → g | tree  | discovered g from f |
| i → h | back  | h is still on the stack when i sees it (h is i's ancestor) |
| g → i | cross | i was already finished before g looked at it |
| c → b | tree  | discovered b from c |
| b → f | cross | f was finished in a different (earlier) tree |
| e → g | cross | g already finished, in earlier tree |
| d → e | cross | e already finished, in earlier tree |
| a → d | cross | d in earlier tree |
| a → c | cross | c in earlier tree |

So: **4 tree, 1 back, 6 cross, 0 forward**.

(No forward edges show up here because every non-tree descendant relationship would require the descendant to still be in the same tree as its ancestor — but in this traversal the only ancestor-descendant pair in one tree is i under h, and i→h goes the wrong way, so it's back, not forward.)

## Problem 3: Table of π, d, f for each vertex

| v | π(v) | d(v) | f(v) |
|---|------|------|------|
| a | NIL  | 17   | 18   |
| b | c    | 14   | 15   |
| c | NIL  | 13   | 16   |
| d | NIL  | 11   | 12   |
| e | NIL  | 9    | 10   |
| f | h    | 2    | 7    |
| g | f    | 5    | 6    |
| h | NIL  | 1    | 8    |
| i | f    | 3    | 4    |

Quick check on parenthesis property: h's interval [1,8] contains f's [2,7], which contains both i's [3,4] and g's [5,6]. c's [13,16] contains b's [14,15]. All other intervals are disjoint. Looks consistent.

# Thank you : D (too lazy to try part -3) Hopefully tomorrow. 